In [20]:
import json
import pandas as pd
from pathlib import Path

# Path to the evaluation output
output_dir = Path("/home/v-murongma/code/OpenHands_SWE-Bench-Optimized/evaluation/evaluation_outputs/outputs/SWE-Gym__SWE-Gym-train/CodeActAgent/Qwen3-Coder-480B-A35B-Instruct_maxiter_100_N_v0.61.0-no-hint-train-qwen3_coder_480b_a35b_instruct-t05-run_1")
eval_file = output_dir / "output.swebench_eval.jsonl"

# Load the evaluation results
results = []
with open(eval_file, 'r') as f:
    for line in f:
        results.append(json.loads(line))

print(f"Total instances: {len(results)}")
print(f"Evaluation file: {eval_file}")


Total instances: 158
Evaluation file: /home/v-murongma/code/OpenHands_SWE-Bench-Optimized/evaluation/evaluation_outputs/outputs/SWE-Gym__SWE-Gym-train/CodeActAgent/Qwen3-Coder-480B-A35B-Instruct_maxiter_100_N_v0.61.0-no-hint-train-qwen3_coder_480b_a35b_instruct-t05-run_1/output.swebench_eval.jsonl


In [27]:
inspect_id = 2
r = results[inspect_id]
instance_id = r['instance_id']
report = r['test_result'].get('report', {})

print(f"Inspecting instance_id: {instance_id}")
print(results[inspect_id].keys())
print(results[inspect_id]['test_result'].keys())
print(results[inspect_id]['test_result'].get('report', {}).keys())
print(results[inspect_id]['test_result']['test_output'])

inspect_id = [r['instance_id'] for r in results].index('getmoto__moto-5706')
print(f"Inspecting instance_id: {instance_id}")
print(results[inspect_id].keys())
print(results[inspect_id]['test_result'].keys())
print(results[inspect_id]['test_result'].get('report', {}).keys())
print(results[inspect_id]['test_result']['test_output'])

Inspecting instance_id: getmoto__moto-5767
dict_keys(['instance_id', 'test_result', 'instruction', 'metadata', 'history', 'metrics', 'error', 'instance'])
dict_keys(['git_patch', 'report', 'apply_patch_output', 'test_output'])
dict_keys(['empty_generation', 'resolved', 'failed_apply_patch', 'error_eval', 'test_timeout'])
+ source /opt/miniconda3/bin/activate
++ _CONDA_ROOT=/opt/miniconda3
++ . /opt/miniconda3/etc/profile.d/conda.sh
+++ export CONDA_EXE=/opt/miniconda3/bin/conda
+++ CONDA_EXE=/opt/miniconda3/bin/conda
+++ export _CE_M=
+++ _CE_M=
+++ export _CE_CONDA=
+++ _CE_CONDA=
+++ export CONDA_PYTHON_EXE=/opt/miniconda3/bin/python
+++ CONDA_PYTHON_EXE=/opt/miniconda3/bin/python
+++ '[' -z x ']'
++ conda activate
++ local cmd=activate
++ case "$cmd" in
++ __conda_activate activate
++ '[' -n '' ']'
++ local ask_conda
+++ PS1=
+++ __conda_exe shell.posix activate
+++ /opt/miniconda3/bin/conda shell.posix activate
++ ask_conda='. "/openhands/micromamba/envs/openhands/etc/conda/deactiv

In [9]:
# Extract key metrics from reports
data = []
for r in results:
    instance_id = r['instance_id']
    test_result = r['test_result']
    report = test_result.get('report', {})
    
    # Check if patch exists by looking at git_patch field
    git_patch = test_result.get('git_patch', '')
    patch_exists = git_patch is not None and len(str(git_patch).strip()) > 0
    
    # Check if patch applied by looking for success indicators in apply_patch_output
    apply_output = test_result.get('apply_patch_output', '')
    patch_applied = 'APPLY_PATCH_PASS' in apply_output if apply_output else False
    
    data.append({
        'instance_id': instance_id,
        'resolved': report.get('resolved', False),
        'patch_exists': patch_exists,
        'patch_applied': patch_applied,
        'empty_gen': report.get('empty_generation', False),
        'failed_apply': report.get('failed_apply_patch', False),
        'error_eval': report.get('error_eval', False),
        'timeout': report.get('test_timeout', False)
    })

df = pd.DataFrame(data)
print(f"\nSummary Statistics:")
print(f"Resolved: {df['resolved'].sum()} / {len(df)} ({df['resolved'].mean()*100:.1f}%)")
print(f"Patch exists: {df['patch_exists'].sum()}")
print(f"Patch applied: {df['patch_applied'].sum()}")
print(f"Empty generation: {df['empty_gen'].sum()}")
print(f"Failed apply: {df['failed_apply'].sum()}")
print(f"Eval errors: {df['error_eval'].sum()}")
print(f"Timeouts: {df['timeout'].sum()}")

# Show the DataFrame
df.head()


Summary Statistics:
Resolved: 35 / 158 (22.2%)
Patch exists: 158
Patch applied: 158
Empty generation: 0
Failed apply: 0
Eval errors: 0
Timeouts: 3


,instance_id,resolved,patch_exists,patch_applied,empty_gen,failed_apply,error_eval,timeout
0,getmoto__moto-7580,True,True,True,False,False,False,False
1,getmoto__moto-5486,False,True,True,False,False,False,False
2,getmoto__moto-5767,False,True,True,False,False,False,False
3,python__mypy-10775,False,True,True,False,False,False,False
4,python__mypy-9629,False,True,True,False,False,False,False


In [10]:
# Analyze different failure patterns
print("="*60)
print("FAILURE ANALYSIS")
print("="*60)

# Patch applied but not resolved (tests failed)
tests_failed = df[(df['patch_applied'] == True) & (df['resolved'] == False)]
print(f"\n1. Patch applied but tests failed: {len(tests_failed)}")
if len(tests_failed) > 0:
    print(f"   Examples: {tests_failed['instance_id'].head(3).tolist()}")

# Patch exists but failed to apply
apply_failed = df[(df['patch_exists'] == True) & (df['patch_applied'] == False) & (df['failed_apply'] == True)]
print(f"\n2. Patch failed to apply: {len(apply_failed)}")
if len(apply_failed) > 0:
    print(f"   Examples: {apply_failed['instance_id'].head(3).tolist()}")

# No patch generated
no_patch = df[df['patch_exists'] == False]
print(f"\n3. No patch generated: {len(no_patch)}")
if len(no_patch) > 0:
    print(f"   Examples: {no_patch['instance_id'].head(3).tolist()}")

# Timeouts
timeouts = df[df['timeout'] == True]
print(f"\n4. Evaluation timeouts: {len(timeouts)}")
if len(timeouts) > 0:
    print(f"   Examples: {timeouts['instance_id'].tolist()}")

# Errors during evaluation
errors = df[df['error_eval'] == True]
print(f"\n5. Evaluation errors: {len(errors)}")
if len(errors) > 0:
    print(f"   Examples: {errors['instance_id'].tolist()}")

FAILURE ANALYSIS

1. Patch applied but tests failed: 123
   Examples: ['getmoto__moto-5486', 'getmoto__moto-5767', 'python__mypy-10775']

2. Patch failed to apply: 0

3. No patch generated: 0

4. Evaluation timeouts: 3
   Examples: ['modin-project__modin-6638', 'modin-project__modin-6821', 'modin-project__modin-6790']

5. Evaluation errors: 0


In [11]:
# Inspect a specific instance
def inspect_instance(instance_id):
    """Show detailed information for a specific instance"""
    for r in results:
        if r['instance_id'] == instance_id:
            test_result = r['test_result']
            report = test_result.get('report', {})
            
            print(f"Instance: {instance_id}")
            print(f"\nReport: {report}")
            
            # Show git_patch info
            git_patch = test_result.get('git_patch', '')
            print(f"\nPatch exists: {git_patch is not None and len(str(git_patch).strip()) > 0}")
            print(f"Patch length: {len(str(git_patch))} chars")
            
            # Show apply_patch_output
            apply_output = test_result.get('apply_patch_output', '')
            if apply_output:
                print(f"\nApply patch output (last 500 chars):")
                print(apply_output[-500:])
            
            # Check test_output
            test_output = test_result.get('test_output', '')
            if test_output:
                has_applied_patch = 'applied patch' in test_output.lower()
                print(f"\nTest output exists: True ({len(test_output)} chars)")
                print(f"Contains 'applied patch': {has_applied_patch}")
            else:
                print(f"\nTest output exists: False")
            
            return r
    print(f"Instance {instance_id} not found")
    return None

# Example: inspect the hydra instance
inspect_instance('facebookresearch__hydra-1531')

Instance: facebookresearch__hydra-1531

Report: {'empty_generation': False, 'resolved': False, 'failed_apply_patch': False, 'error_eval': False, 'test_timeout': False}

Patch exists: True
Patch length: 18860 chars

Apply patch output (last 500 chars):
onf/__init__.py cleanly.
Applied patch hydra/conf/hydra/env/default.yaml cleanly.
Applied patch hydra/conf/hydra/env/production.yaml cleanly.
Applied patch hydra/conf/hydra/env/with_callbacks.yaml cleanly.
Applied patch test_env_config_group.py cleanly.
Applied patch test_env_functionality.py cleanly.
Applied patch test_env_integration.py cleanly.
Applied patch test_env_integration_clean.py cleanly.
APPLY_PATCH_PASS

Test output exists: True (30047 chars)
Contains 'applied patch': False


{'instance_id': 'facebookresearch__hydra-1531',
 'test_result': {'git_patch': 'diff --git a/demo_env_config_group.py b/demo_env_config_group.py\nnew file mode 100644\nindex 0000000000..c38787aba0\n--- /dev/null\n+++ b/demo_env_config_group.py\n@@ -0,0 +1,85 @@\n+#!/usr/bin/env python3\n+"""\n+Demo script showing that the env config group works correctly.\n+"""\n+\n+import sys\n+import os\n+sys.path.insert(0, \'/workspace/facebookresearch__hydra__1.1\')\n+\n+from hydra.core.config_store import ConfigStore\n+from hydra.conf import HydraConf\n+from omegaconf import OmegaConf\n+\n+\n+def demo_env_config_group():\n+    """Demonstrate that the env config group is properly implemented"""\n+    print("=== Demo: Env Config Group Implementation ===")\n+    \n+    # Show that env config group is in the defaults\n+    hydra_conf = HydraConf()\n+    print("HydraConf defaults:")\n+    for i, default in enumerate(hydra_conf.defaults):\n+        print(f"  {i+1}. {default}")\n+        \n+    # Check sp

In [12]:
# Show test output excerpt for analysis
def show_test_output(instance_id, head=500, tail=500):
    """Show beginning and end of test output"""
    for r in results:
        if r['instance_id'] == instance_id:
            test_output = r['test_result'].get('test_output', '')
            print(f"Instance: {instance_id}")
            print(f"Test output length: {len(test_output)} chars")
            print(f"\n{'='*60}")
            print("BEGINNING:")
            print(test_output[:head])
            print(f"\n{'='*60}")
            print("END:")
            print(test_output[-tail:])
            return
    print(f"Instance {instance_id} not found")

## Quick Reference

**Key Functions:**
- `inspect_instance(instance_id)` - Show detailed info for an instance
- `show_test_output(instance_id, head=500, tail=500)` - Show test output excerpts
- `df` - DataFrame with all results for filtering/analysis

**Example Usage:**
```python
# Show all unresolved cases
df[df['resolved'] == False]

# Inspect a specific case
inspect_instance('your-instance-id')

# Show test output
show_test_output('your-instance-id')
```

In [ ]:
# Analyze test status details
def analyze_test_status(instance_id):
    """Show detailed test status if available"""
    for r in results:
        if r['instance_id'] == instance_id:
            test_result = r['test_result']
            report = test_result.get('report', {})
            
            print(f"Instance: {instance_id}")
            print(f"Resolved: {report.get('resolved', False)}")
            
            # Check for detailed test status
            tests_status = report.get('tests_status', {})
            if tests_status:
                print(f"\n{'='*60}")
                print("DETAILED TEST STATUS (from SWE-Bench grading):")
                print(f"{'='*60}")
                
                for category, status in tests_status.items():
                    if isinstance(status, dict):
                        success = status.get('success', [])
                        failure = status.get('failure', [])
                        if success or failure:
                            print(f"\n{category}:")
                            if success:
                                print(f"  ✓ Success ({len(success)}): {success[:3]}")
                                if len(success) > 3:
                                    print(f"    ... and {len(success) - 3} more")
                            if failure:
                                print(f"  ✗ Failure ({len(failure)}): {failure[:3]}")
                                if len(failure) > 3:
                                    print(f"    ... and {len(failure) - 3} more")
            else:
                print("\nNo detailed test status available")
                print("This happens when 'patch_successfully_applied' is False")
                print("The grading script didn't parse test results because:")
                print("  - The phrase 'applied patch' was not found in test_output")
                
                # Check if we can still see test output
                test_output = test_result.get('test_output', '')
                if test_output:
                    print(f"\nHowever, test_output EXISTS ({len(test_output)} chars)")
                    print("The patch likely applied, but grading heuristic failed")
                    
                    # Try to find test results manually
                    if 'FAILED' in test_output or 'PASSED' in test_output:
                        print("\nTest execution indicators found in output:")
                        if 'FAILED' in test_output:
                            print("  - Contains 'FAILED'")
                        if 'PASSED' in test_output:
                            print("  - Contains 'PASSED'")
                        if 'passed' in test_output.lower():
                            import re
                            # Look for pytest summary
                            matches = re.findall(r'(\d+) passed', test_output, re.IGNORECASE)
                            if matches:
                                print(f"  - Found: {matches[-1]} passed")
                        if 'failed' in test_output.lower():
                            matches = re.findall(r'(\d+) failed', test_output, re.IGNORECASE)
                            if matches:
                                print(f"  - Found: {matches[-1]} failed")
            
            return r
    print(f"Instance {instance_id} not found")
    return None

# Test with a resolved case
print("Example 1: Resolved case with test status")
analyze_test_status('getmoto__moto-5601')

print("\n" + "="*80 + "\n")

# Test with the problematic hydra case
print("Example 2: Case with patch_successfully_applied=False")
analyze_test_status('facebookresearch__hydra-1531')

In [ ]:
# Create a comprehensive summary with test status
def create_detailed_summary():
    """Create a DataFrame with test status information"""
    data = []
    for r in results:
        instance_id = r['instance_id']
        test_result = r['test_result']
        report = test_result.get('report', {})
        tests_status = report.get('tests_status', {})
        
        # Count test results
        f2p_success = len(tests_status.get('FAIL_TO_PASS', {}).get('success', []))
        f2p_failure = len(tests_status.get('FAIL_TO_PASS', {}).get('failure', []))
        p2p_success = len(tests_status.get('PASS_TO_PASS', {}).get('success', []))
        p2p_failure = len(tests_status.get('PASS_TO_PASS', {}).get('failure', []))
        
        has_test_status = tests_status != {}
        
        data.append({
            'instance_id': instance_id,
            'resolved': report.get('resolved', False),
            'has_test_status': has_test_status,
            'F2P_success': f2p_success,
            'F2P_failure': f2p_failure,
            'P2P_success': p2p_success,
            'P2P_failure': p2p_failure,
            'total_tests': f2p_success + f2p_failure + p2p_success + p2p_failure
        })
    
    detailed_df = pd.DataFrame(data)
    
    print("Summary by test status availability:")
    print(f"With test status: {detailed_df['has_test_status'].sum()}")
    print(f"Without test status: {(~detailed_df['has_test_status']).sum()}")
    print(f"\nOf those WITH test status:")
    print(f"  Resolved: {detailed_df[detailed_df['has_test_status']]['resolved'].sum()}")
    print(f"  Not resolved: {(~detailed_df[detailed_df['has_test_status']]['resolved']).sum()}")
    print(f"\nOf those WITHOUT test status:")
    print(f"  Resolved: {detailed_df[~detailed_df['has_test_status']]['resolved'].sum()}")
    print(f"  Not resolved: {(~detailed_df[~detailed_df['has_test_status']]['resolved']).sum()}")
    
    return detailed_df

detailed_df = create_detailed_summary()
detailed_df.head(10)

In [ ]:
# Manually parse test results from test_output when grading fails
import re

def parse_test_output_manually(test_output):
    """
    Manually parse pytest output to extract test results
    This is what SWE-Gym grading does, but only when 'patch_successfully_applied' is True
    """
    if not test_output:
        return None
    
    # Different repositories use different test parsers
    # Most use pytest, let's parse pytest output
    
    results = {
        'passed': [],
        'failed': [],
        'summary': None
    }
    
    # Look for pytest test results (format: PASSED test_file.py::test_name)
    passed_matches = re.findall(r'PASSED\s+([\w\./]+\.py::\S+)', test_output)
    failed_matches = re.findall(r'FAILED\s+([\w\./]+\.py::\S+)', test_output)
    
    results['passed'] = passed_matches
    results['failed'] = failed_matches
    
    # Look for summary line (e.g., "1 failed, 527 passed, 121 skipped")
    summary_pattern = r'=+\s*(.*?)\s*=+\s*$'
    lines = test_output.split('\n')
    for line in reversed(lines):  # Check from end
        if 'failed' in line.lower() or 'passed' in line.lower():
            # Extract numbers
            num_failed = re.findall(r'(\d+)\s+failed', line)
            num_passed = re.findall(r'(\d+)\s+passed', line)
            num_skipped = re.findall(r'(\d+)\s+skipped', line)
            
            if num_failed or num_passed:
                results['summary'] = {
                    'failed': int(num_failed[0]) if num_failed else 0,
                    'passed': int(num_passed[0]) if num_passed else 0,
                    'skipped': int(num_skipped[0]) if num_skipped else 0
                }
                break
    
    return results

# Test with the hydra case
def analyze_with_manual_parsing(instance_id):
    """Analyze instance with manual test parsing when grading fails"""
    for r in results:
        if r['instance_id'] == instance_id:
            test_result = r['test_result']
            report = test_result.get('report', {})
            tests_status = report.get('tests_status', {})
            
            print(f"Instance: {instance_id}")
            print(f"Resolved: {report.get('resolved', False)}")
            print(f"Has tests_status from grading: {bool(tests_status)}")
            
            # If no tests_status, try manual parsing
            if not tests_status:
                test_output = test_result.get('test_output', '')
                if test_output:
                    print(f"\nTest output exists ({len(test_output)} chars)")
                    print("Attempting manual parsing...")
                    
                    parsed = parse_test_output_manually(test_output)
                    if parsed:
                        print(f"\n{'='*60}")
                        print("MANUALLY PARSED TEST RESULTS:")
                        print(f"{'='*60}")
                        
                        if parsed['summary']:
                            print(f"\nSummary:")
                            print(f"  Passed: {parsed['summary']['passed']}")
                            print(f"  Failed: {parsed['summary']['failed']}")
                            print(f"  Skipped: {parsed['summary']['skipped']}")
                        
                        if parsed['passed']:
                            print(f"\nPassed tests ({len(parsed['passed'])}):")
                            for test in parsed['passed'][:5]:
                                print(f"  ✓ {test}")
                            if len(parsed['passed']) > 5:
                                print(f"  ... and {len(parsed['passed']) - 5} more")
                        
                        if parsed['failed']:
                            print(f"\nFailed tests ({len(parsed['failed'])}):")
                            for test in parsed['failed'][:5]:
                                print(f"  ✗ {test}")
                            if len(parsed['failed']) > 5:
                                print(f"  ... and {len(parsed['failed']) - 5} more")
                    else:
                        print("Could not parse test results")
            else:
                print("\nTests status from SWE-Gym grading:")
                for category, status in tests_status.items():
                    if isinstance(status, dict):
                        success = status.get('success', [])
                        failure = status.get('failure', [])
                        if success or failure:
                            print(f"\n{category}:")
                            if success:
                                print(f"  ✓ Success: {len(success)}")
                            if failure:
                                print(f"  ✗ Failure: {len(failure)}")
            
            return r
    print(f"Instance {instance_id} not found")
    return None

# Test with both cases
print("Case 1: With grading (patch_successfully_applied=True)")
analyze_with_manual_parsing('getmoto__moto-5601')

print("\n" + "="*80 + "\n")

print("Case 2: Without grading (patch_successfully_applied=False)")
analyze_with_manual_parsing('facebookresearch__hydra-1531')

In [ ]:
# Create enhanced DataFrame with manual parsing for all instances
def create_enhanced_summary():
    """Create DataFrame with both grading and manual parsing results"""
    data = []
    
    for r in results:
        instance_id = r['instance_id']
        test_result = r['test_result']
        report = test_result.get('report', {})
        tests_status = report.get('tests_status', {})
        
        # Get from grading if available
        has_grading = bool(tests_status)
        resolved = report.get('resolved', False)
        
        # Initialize counts
        f2p_success = 0
        f2p_failure = 0
        total_passed = 0
        total_failed = 0
        
        if has_grading:
            # Use grading results
            f2p_success = len(tests_status.get('FAIL_TO_PASS', {}).get('success', []))
            f2p_failure = len(tests_status.get('FAIL_TO_PASS', {}).get('failure', []))
            p2p_success = len(tests_status.get('PASS_TO_PASS', {}).get('success', []))
            p2p_failure = len(tests_status.get('PASS_TO_PASS', {}).get('failure', []))
            total_passed = f2p_success + p2p_success
            total_failed = f2p_failure + p2p_failure
        else:
            # Try manual parsing
            test_output = test_result.get('test_output', '')
            if test_output:
                parsed = parse_test_output_manually(test_output)
                if parsed and parsed['summary']:
                    total_passed = parsed['summary']['passed']
                    total_failed = parsed['summary']['failed']
        
        # Check if patch applied
        apply_output = test_result.get('apply_patch_output', '')
        patch_applied = 'APPLY_PATCH_PASS' in apply_output if apply_output else False
        
        data.append({
            'instance_id': instance_id,
            'resolved': resolved,
            'has_grading': has_grading,
            'patch_applied': patch_applied,
            'F2P_success': f2p_success,
            'F2P_failure': f2p_failure,
            'total_passed': total_passed,
            'total_failed': total_failed,
            'has_test_output': bool(test_result.get('test_output', ''))
        })
    
    enhanced_df = pd.DataFrame(data)
    
    print("Enhanced Summary:")
    print(f"Total instances: {len(enhanced_df)}")
    print(f"\nResolved: {enhanced_df['resolved'].sum()} ({enhanced_df['resolved'].mean()*100:.1f}%)")
    print(f"\nGrading status:")
    print(f"  With grading (patch_successfully_applied=True): {enhanced_df['has_grading'].sum()}")
    print(f"  Without grading (patch_successfully_applied=False): {(~enhanced_df['has_grading']).sum()}")
    print(f"\nPatch application:")
    print(f"  Patch applied (APPLY_PATCH_PASS): {enhanced_df['patch_applied'].sum()}")
    print(f"  Has test output: {enhanced_df['has_test_output'].sum()}")
    
    # Show cases where patch applied but no grading
    no_grading_but_applied = enhanced_df[enhanced_df['patch_applied'] & ~enhanced_df['has_grading']]
    print(f"\n⚠️  Cases with patch applied but no grading: {len(no_grading_but_applied)}")
    if len(no_grading_but_applied) > 0:
        print(f"   (These are FALSE NEGATIVES - patch applied successfully)")
        print(f"   Total passed tests (manual parsing): {no_grading_but_applied['total_passed'].sum()}")
        print(f"   Total failed tests (manual parsing): {no_grading_but_applied['total_failed'].sum()}")
    
    return enhanced_df

enhanced_df = create_enhanced_summary()
print("\n" + "="*60)
print("Sample of cases without grading but with test results:")
no_grading = enhanced_df[~enhanced_df['has_grading'] & (enhanced_df['total_passed'] > 0)]
no_grading.head(10)

## Understanding the False Negative Issue

**The Problem:**
When the SWE-Gym grading script doesn't find the phrase "applied patch" (lowercase) in the test_output, it sets `patch_successfully_applied=False` and **stops processing** - it won't parse the test results even though:
1. The patch WAS applied (you see `APPLY_PATCH_PASS`)
2. Tests DID run (test_output contains results)
3. The only issue is the grading heuristic failed

**The Solution:**
The cells above manually parse the test_output using the same pytest parsing logic that SWE-Gym uses, allowing us to see test results even for false negative cases.

**Key Functions:**
- `parse_test_output_manually(test_output)` - Parses pytest output manually
- `analyze_with_manual_parsing(instance_id)` - Shows both grading and manual results
- `enhanced_df` - DataFrame with manual parsing for all instances